In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
%autosave 300

In [ ]:
import os

os.chdir("../")
print(os.getcwd())

#### Multiclass XOR Classification Example

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchmetrics
import torchvision.utils as vutils
from PIL import Image
from torchvision import transforms
import torchvision

In [ ]:
data = pd.read_csv("data/xor.csv")
data

In [ ]:
data["class label"].value_counts()

In [ ]:
X = data[["x1", "x2"]].values
y = data["class label"].values

print(X.shape, y.shape)

In [ ]:
# split into train, validation and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)
print(np.bincount(y_train), np.bincount(y_val), np.bincount(y_test))

In [ ]:
plt.plot(
    X_train[y_train == 0, 0],
    X_train[y_train == 0, 1],
    marker="D",
    markersize=10,
    linestyle="",
    label="Class 0",
)

plt.plot(
    X_train[y_train == 1, 0],
    X_train[y_train == 1, 1],
    marker="^",
    markersize=13,
    linestyle="",
    label="Class 1",
)

plt.legend(loc=2)

plt.xlim([-5, 5])
plt.ylim([-5, 5])

plt.xlabel("Feature $x_1$", fontsize=12)
plt.ylabel("Feature $x_2$", fontsize=12)

plt.grid()
plt.show()

In [ ]:
class XorModel(nn.Module):
    def __init__(self, in_features, hidden_units, out_features):
        super(XorModel, self).__init__()
        self.layer_1 = nn.Linear(in_features, hidden_units)
        self.layer_2 = nn.Linear(hidden_units, 15)
        self.layer_3 = nn.Linear(15, out_features)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        x = self.relu(x)
        x = self.layer_3(x)
        return x

In [ ]:
class XORDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __getitem__(self, index):
        x = self.X[index]
        y = self.y[index]
        return x, y

    def __len__(self):
        return self.X.shape[0]

In [ ]:
train_ds = XORDataset(X_train, y_train)
val_ds = XORDataset(X_val, y_val)
test_ds = XORDataset(X_test, y_test)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=32,
    shuffle=True,
)

val_loader = DataLoader(
    dataset=val_ds,
    batch_size=32,
    shuffle=False,
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=32,
    shuffle=False,
)

In [ ]:
# for X_batch, y_batch in train_loader:
#     print(X_batch.shape, y_batch.shape)
#     break

# for X_batch, y_batch in val_loader:
#     print(X_batch.shape, y_batch.shape)
#     break

# for X_batch, y_batch in test_loader:
#     print(X_batch.shape, y_batch.shape)
#     break

In [ ]:
metric = torchmetrics.Accuracy(task="multiclass", num_classes=2)

In [ ]:
def training_loop(model, train_loader, criterion, optimizer, num_epochs, metric):
    """Training loop for the model."""
    for epoch in range(num_epochs):
        model = model.train()
        for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
            # Forward pass
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            pred_labels = preds.argmax(dim=1)
            metric.update(pred_labels, y_batch)

        # calculate metrics
        accuracy = metric.compute()
        metric.reset()
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}")

In [ ]:
def evaluate_model(model, test_loader, criterion, metric):
    """Evaluate the model on the test set."""
    model = model.eval()
    test_loss = 0.0
    with torch.inference_mode():
        for X_batch, y_batch in test_loader:
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            test_loss += loss.item()
            pred_labels = preds.argmax(dim=1)
            metric.update(pred_labels, y_batch)

    accuracy = metric.compute()
    metric.reset()
    test_loss /= len(test_loader)
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {accuracy:.4f}")

In [ ]:
model = XorModel(in_features=2, hidden_units=8, out_features=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

In [ ]:
training_loop(model, train_loader, criterion, optimizer, num_epochs=200, metric=metric)

In [ ]:
evaluate_model(model, train_loader, criterion, metric)
evaluate_model(model, val_loader, criterion, metric)
evaluate_model(model, test_loader, criterion, metric)

In [ ]:
from matplotlib.colors import ListedColormap
import numpy as np


def plot_decision_regions(X, y, classifier, resolution=0.02):

    # setup marker generator and color map
    markers = ("D", "^", "x", "s", "v")
    colors = ("C0", "C1", "C2", "C3", "C4")
    cmap = ListedColormap(colors[: len(np.unique(y))])

    # plot the decision surface
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, resolution), np.arange(x2_min, x2_max, resolution))
    tensor = torch.tensor(np.array([xx1.ravel(), xx2.ravel()]).T).float()
    logits = classifier.forward(tensor)
    Z = np.argmax(logits.detach().numpy(), axis=1)

    Z = Z.reshape(xx1.shape)
    plt.contourf(xx1, xx2, Z, alpha=0.4, cmap=cmap)
    plt.xlim(xx1.min(), xx1.max())
    plt.ylim(xx2.min(), xx2.max())

    # plot class samples
    for idx, cl in enumerate(np.unique(y)):
        plt.scatter(
            x=X[y == cl, 0],
            y=X[y == cl, 1],
            alpha=0.8,
            color=cmap(idx),
            # edgecolor='black',
            marker=markers[idx],
            label=cl,
        )

In [ ]:
plot_decision_regions(X_train, y_train, classifier=model)

##### MNIST Classification Example

In [ ]:
import os
from git import Repo

if not os.path.exists("mnist-pngs"):
    Repo.clone_from("https://github.com/rasbt/mnist-pngs", "mnist-pngs")

In [ ]:
df_train = pd.read_csv("mnist-pngs/train.csv")
df_train.head()

In [ ]:
df_test = pd.read_csv("mnist-pngs/test.csv")
df_test.head()

In [ ]:
df_train = pd.read_csv('mnist-pngs/train.csv')
df_train = df_train.sample(frac=1, random_state=123)

loc = round(df_train.shape[0]*0.9)
df_new_train = df_train.iloc[:loc]
df_new_val = df_train.iloc[loc:]

df_new_train.to_csv('mnist-pngs/new_train.csv', index=None)
df_new_val.to_csv('mnist-pngs/new_val.csv', index=None)

In [ ]:
class MnistDataset(Dataset):
    """
    Custom Dataset for loading MNIST images and labels from a CSV file.
    The CSV file should contain two columns: 'filepath' and 'label'.
    'filepath' contains the relative path to the image file.
    'label' contains the corresponding class label for the image.
    """
    def __init__(self,csv_path, img_dir, transform=None):
        df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

        self.img_names = df['filepath']
        self.labels = df['label']
    
    def __getitem__(self, index):
        img = Image.open(os.path.join(self.img_dir, self.img_names[index]))

        if self.transform is not None:
            img = self.transform(img)

        label = self.labels[index]
        return img, label
    
    def __len__(self):
        return self.labels.shape[0]

In [ ]:
def viz_batch_images(batch):

    plt.figure(figsize=(8, 8))
    plt.axis("off")
    plt.title("Training images")
    plt.imshow(
        np.transpose(
            vutils.make_grid(batch[0][:64], padding=2, normalize=True), (1, 2, 0)
        )
    )
    plt.show()

In [ ]:
data_transforms = {
        "train": transforms.Compose(
            [
                transforms.Resize(32),
                transforms.CenterCrop((28, 28)),
                transforms.ToTensor(),
                transforms.Normalize((0.5,), (0.5,)),
            ]
        ),
        "test": transforms.Compose(
            [
                transforms.Resize(32),
                transforms.CenterCrop((28, 28)),
                transforms.ToTensor(),
                transforms.Normalize((0.5,), (0.5,)),
            ]
        ),
    }

In [ ]:
train_dataset = MnistDataset(
    csv_path = "mnist-pngs/new_train.csv",
    img_dir = "mnist-pngs",
    transform = data_transforms["train"]
)

val_dataset = MnistDataset(
    csv_path = "mnist-pngs/new_val.csv",
    img_dir = "mnist-pngs",
    transform = data_transforms["test"]
)

test_dataset = MnistDataset(
    csv_path = "mnist-pngs/test.csv",
    img_dir = "mnist-pngs",
    transform = data_transforms["test"]
)

In [ ]:
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False,
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
)

In [ ]:
batch = next(iter(train_dataloader))
viz_batch_images(batch[0])

In [ ]:
for X_batch, y_batch in train_dataloader:
    print(X_batch.shape, y_batch.shape)
    break

# for X_batch, y_batch in val_dataloader:
#     print(X_batch.shape, y_batch.shape)
#     break

# for X_batch, y_batch in test_dataloader:
#     print(X_batch.shape, y_batch.shape)
#     break

In [ ]:
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Training images")
plt.imshow(np.transpose(torchvision.utils.make_grid(
    X_batch[:64], 
    padding=1,
    pad_value=1.0,
    normalize=True),
    (1, 2, 0)))
plt.show()

In [ ]:
class MnistModel(nn.Module):
    """
    Simple Feedforward Neural Network for MNIST classification.
    """
    def __init__(self, in_features, hidden_units, out_features):
        super(MnistModel, self).__init__()
        self.layer_1 = nn.Linear(in_features, hidden_units)
        self.layer_2 = nn.Linear(hidden_units, 64)
        self.layer_3 = nn.Linear(64, out_features)
        self.relu = nn.ReLU()
    
    def forward(self,x):
        x= torch.flatten(x,start_dim=1)
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        x = self.relu(x)
        x = self.layer_3(x)
        return x

In [ ]:
metric = torchmetrics.Accuracy(task="multiclass", num_classes=10)

In [ ]:
model = MnistModel(in_features=28*28, hidden_units=128, out_features=10)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

In [ ]:
training_loop(model, train_dataloader, criterion, optimizer, num_epochs=10, metric=metric)

In [ ]:
evaluate_model(model, train_dataloader, criterion, metric)
evaluate_model(model, val_dataloader, criterion, metric)
evaluate_model(model, test_dataloader, criterion, metric)

#### Using MPS backend for Mac Users

In [ ]:
print(f"torch backend MPS is available? {torch.backends.mps.is_available()}")
print(f"current PyTorch installation built with MPS activated? {torch.backends.mps.is_built()}")

In [ ]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
def training_loop_device(model, train_loader, criterion, optimizer, num_epochs, metric, device):
    """Training loop for the model."""
    metric = metric.to(device)
    for epoch in range(num_epochs):
        model = model.train()
        for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            # Forward pass
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            pred_labels = preds.argmax(dim=1)
            metric.update(pred_labels, y_batch)

        # calculate metrics
        accuracy = metric.compute()
        metric.reset()
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}")

In [ ]:
def evaluate_model_device(model, test_loader, criterion, metric, device):
    """Evaluate the model on the test set."""
    metric = metric.to(device)
    model = model.eval()
    test_loss = 0.0
    with torch.inference_mode():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            test_loss += loss.item()
            pred_labels = preds.argmax(dim=1)
            metric.update(pred_labels, y_batch)

    accuracy = metric.compute()
    metric.reset()
    test_loss /= len(test_loader)
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {accuracy:.4f}")

In [ ]:
metric = torchmetrics.Accuracy(task="multiclass", num_classes=10)
model = MnistModel(in_features=28*28, hidden_units=128, out_features=10)
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

In [ ]:
training_loop_device(model, train_dataloader, criterion, optimizer, num_epochs=10, metric=metric, device=device)

In [ ]:
evaluate_model_device(model, train_dataloader, criterion, metric, device=device)
evaluate_model_device(model, val_dataloader, criterion, metric, device=device)
evaluate_model_device(model, test_dataloader, criterion, metric, device=device)

#### Few Impotant Python Additonal Helpers